## Analysing first principle results

In [ ]:
import pandas as pd
import json
import os
import numpy as np; np.random.seed(42)
from glob import glob
from tqdm import tqdm
import seaborn as sns
import pdb

import textalloc as ta

import sys; sys.path.append('..')
from pareto_utils import front

import matplotlib.pyplot as plt
import matplotlib

matplotlib.rc('pdf', fonttype=42)
matplotlib.rc('ps', fonttype=42)

params = {'mathtext.default': 'regular' }          
plt.rcParams.update(params)

sns.set(font_scale=1.25, style='whitegrid')
sns.set(palette='magma')
sns.set_style("whitegrid")

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Nimbus Roman', 'Times New Roman', 'Liberation Serif', 'DejaVu Serif'],
})

rdir  = '../../results/first-principles-tuning/'
figdir = f'../figs/first-principles-tuning/'
ddir = '../../datasets/firstprinciples'


def save(name='tmp',h=None,pad=1.08,w_pad=None):
    name = name.strip().replace(' ','-').replace('%','pct')
    if h == None:
        h = plt.gcf()

    h.tight_layout(pad=pad, w_pad=w_pad)

    print('saving',figdir+'/'+name+'.pdf')

    if not os.path.exists(figdir):
        os.makedirs(figdir)

    plt.savefig(figdir+'/'+name+'.pdf', bbox_inches='tight')
    plt.savefig(figdir+'/'+name+'.png', bbox_inches='tight')

# read data from feather

In [ ]:
df_results = pd.read_feather(rdir+'results.feather')
df_sum = pd.read_csv(rdir+'results-summary.csv.gz', compression='gzip')

# -1 marks a failed complexity extraction; mask it to NaN so means/medians skip it
df_results['model_size'] = df_results['model_size'].replace(-1, np.nan)
df_sum['model_size'] = df_sum['model_size'].replace(-1, np.nan)

print('mean trial count:')
print(df_results.groupby('algorithm')['dataset'].count().sort_values()
      / df_results.dataset.nunique())

In [ ]:
from scripts.impute_missing_runs import impute_missing_runs

_impute_missing_runs = True
_impute_r2_source     = 'fixed'  # 'fixed' or 'linear'
_impute_r2_test       = 0.0
_impute_model_size    = "dataset_median"  # 'dataset_median', 'algorithm_median', 'algorithm_mean', 'linear_expression_size', or a fixed number
_impute_algorithms    = None
_linear_algorithm_name = 'Linear'

# df_results_imputed / df_sum_imputed are for analyses that need a complete panel; the Pareto plots use the unimputed frames

if _impute_missing_runs:
    df_results_imputed, df_sum_imputed = impute_missing_runs(
        df_results, df_sum,
        datadir=ddir,
        impute_r2_source=_impute_r2_source,
        impute_r2_test=_impute_r2_test,
        impute_model_size=_impute_model_size,
        impute_algorithms=_impute_algorithms,
        linear_algorithm_name=_linear_algorithm_name,
    )
else:
    df_results_imputed = df_results
    df_sum_imputed = df_sum


# Pareto plot

Plot the method results as trade-offs between accuracy and simplicity

In [ ]:
def bootstrap(val, n = 1000, fn=np.mean):
    val_samples = []
    for i in range(n):
        sample = np.random.randint(0,len(val)-1, size=len(val))
        val_samples.append( fn(val[sample]) )

    m = np.mean(val_samples)
    sd = np.std(val_samples)
    ci_upper  = np.quantile(val_samples,0.95)
    ci_lower  = np.quantile(val_samples,0.05)

    return m, sd, ci_upper,ci_lower

In [ ]:
import numpy as np

fs_data = {
    'first_principles_bode': {
        'model_size': 6,
        'r2_zero_test': 1.0,
        'predict': lambda X: 0.4 + 0.3 * np.exp(X[:, 0]),
        'latex_eq': r'$0.4 + 0.3 \cdot e^{x}$',
    },

    'first_principles_hubble': {
        'model_size': 3,
        'r2_zero_test': 1.0,
        'predict': lambda X: 73.3 * X[:, 0],
        'latex_eq': r'$D \cdot H_0$',
    },

    'first_principles_kepler': {
        'model_size': 8,
        'r2_zero_test': 1.0,
        'predict': lambda X: 1000 * np.sqrt(X[:, 0]**3 / 7.5),
        'latex_eq': r'$1000 \cdot \sqrt{\frac{k \cdot a^3}{7.5}}$',
    },

    'first_principles_tully_fisher': {
        'model_size': 5,
        'r2_zero_test': 1.0,
        'predict': lambda X: 26.1 * X[:, 0]**2.5,
        'latex_eq': r'$k \cdot \Delta V^{2.5}$',
    },

    'first_principles_planck': {
        'model_size': 20,
        'r2_zero_test': 1.0,
        'predict': lambda X: (
            2 * 6.62607004e-34 * X[:, 0]**3 / (299792458**2)
            * (1 / (np.exp(6.62607004e-34 * X[:, 0] / (1.38064852e-23 * X[:, 1])) - 1))
        ),
        'latex_eq': r'$\frac{2 \cdot h \cdot \nu^3}{c^2} \cdot \frac{1}{e^{\frac{h \cdot \nu}{k_B \cdot T}} - 1}$',
    },

    'first_principles_ideal_gas': {
        'model_size': 7,
        'r2_zero_test': 1.0,
        'predict': lambda X: X[:, 0] * 8.314 * X[:, 1] / X[:, 2],
        'latex_eq': r'$\frac{n \cdot R \cdot T}{V}$',
    },

    'first_principles_leavitt': {
        'model_size': 5,
        'r2_zero_test': 1.0,
        'predict': lambda X: -2.084 * X[:, 0] + 15.65,
        'latex_eq': r'$\log_{10}(P) \cdot \alpha + \delta$',
    },

    'first_principles_newton': {
        'model_size': 9,
        'r2_zero_test': 1.0,
        'predict': lambda X: 6.67408e-11 * X[:, 1] * X[:, 2] / X[:, 0]**2,
        'latex_eq': r'$\frac{G \cdot m_1 \cdot m_2}{r^2}$',
    },

    'first_principles_rydberg': {
        'model_size': 15,
        'r2_zero_test': 1.0,
        'predict': lambda X: 1 / (1.097e7 * (1 / X[:, 0]**2 - 1 / X[:, 1]**2)),
        'latex_eq': r'$\frac{1}{R_H \cdot \left(\frac{1}{n_1^2} - \frac{1}{n_2^2}\right)}$',
    },

    'first_principles_schechter': {
        'model_size': 13,
        'r2_zero_test': 1.0,
        'predict': lambda X: (0.002 / 2.5e8) * (X[:, 0] / 2.5e8)**-1.20 * np.exp(-X[:, 0] / 2.5e8),
        'latex_eq': r'$\frac{\phi_*}{L_*} \cdot \left(\frac{L}{L_*}\right)^{\alpha} \cdot e^{-\frac{L}{L_*}}$',
    },

    'first_principles_absorption': {
        'model_size': 2,
        'r2_zero_test': 0.987,
        'predict': lambda X: np.log(X[:, 0]),
        'latex_eq': r'$\log{x}$',
    },

    'first_principles_supernovae_zr': {
        'model_size': 14,
        'r2_zero_test': 0.987,
        'latex_eq': r'$\frac{A}{B \cdot e^{C t} + e^{-D t}}$',
    },

    'first_principles_supernovae_zg': {
        'model_size': 1,
        'r2_zero_test': 0.987,
        'latex_eq': r'$\frac{A}{B \cdot e^{C t} + e^{-D t}}$',
    },
}


In [ ]:
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import sys; sys.path.append('../../experiment/')
from read_file import read_file

for ds, attr in fs_data.items():
    if not 'predict' in attr:
        continue
    print("----------------------")
    features, labels, feature_names = read_file(f"{ddir}/{ds}/{ds}.tsv.gz", use_dataframe=False, sep='\t')
    print(feature_names)
    print(features.shape)

    y_pred = attr['predict'](features)

    # Miles generated data have log(y) instead of raw y in the labels column
    if ds in ['first_principles_bode', 'first_principles_ideal_gas', 'first_principles_newton', 'first_principles_planck',
              'first_principles_rydberg', 'first_principles_schechter']:
        y_pred = np.log(y_pred)
    
    if ds == 'first_principles_planck':
        idxs = y_pred != -np.inf
        y_pred = y_pred[idxs]
        labels = labels[idxs]

    print(features[:3])
    print(y_pred[:10])
    print(labels[:10])

    score = r2_score(labels, y_pred)
    
    print(ds, score)
    fs_data[ds]['r2_zero_test'] = np.maximum(0.0, score)

In [ ]:
# raw algorithm name -> display name for plots; unlisted algorithms keep their raw name
algorithm_name_remap = {
    "rasr_94m": "RASR-94M-L",
    "rasr_94m_lambda0.15": "RASR-94M-M",
    "rasr_94m_lambda0.5": "RASR-94M-S",
}

# algorithm(s) to highlight with a diamond in the Pareto plots; may be empty
highlight_algorithms = [] # "rasr_94m_lambda0.15", "rasr_94m_lambda0.5"

In [ ]:
def prepare_pareto_data(
    df_sum,
    estimator=np.median,
    facet_column='dataset',
    xcol='r2_test_rank',
    ycol='model_size_rank',
):
    """Compute aggregated Pareto data and ranks for each dataset."""

    is_max_metric = lambda s: not any(k in s for k in ['rank', 'size', 'mse'])
    x_greater_is_better = is_max_metric(xcol)
    y_greater_is_better = is_max_metric(ycol)

    numeric_cols = df_sum.select_dtypes('number').columns
    grouped = (
        df_sum.groupby(['algorithm', facet_column])[numeric_cols]
        .agg(estimator)
        .reset_index()
    )

    data = grouped[['algorithm', facet_column, xcol, ycol]].copy()
    levels = len(data['algorithm'].unique())
    pareto_ranks = np.ones(len(data)) * levels

    for val in data[facet_column].unique():
        objs = data[data[facet_column] == val].copy()
        if x_greater_is_better:
            objs.loc[:, xcol] = -objs[xcol]
        if y_greater_is_better:
            objs.loc[:, ycol] = -objs[ycol]

        for level in range(levels):
            if not np.any(np.isfinite(objs[[xcol, ycol]])):
                break
            PF = front(objs[xcol].values, objs[ycol].values)
            if len(PF) > 0:
                pareto_ranks[objs.iloc[PF].index] = level
            objs.loc[objs.iloc[PF].index, [xcol, ycol]] = np.inf

    data['pareto_rank'] = pareto_ranks
    return data


def plot_pareto_facets(
    df_sum,
    data,
    estimator=np.median,
    plot_error=True,
    levels_to_show=1,
    facet_column='dataset',
    xcol='r2_test_rank',
    ycol='model_size_rank',
    name_remap=None,
    highlight_algorithms=None,
    col_wrap=6,
    xlabel=None,
    ylabel=None,
    height=3,
    aspect=1,
    save_pad=1.08,
    save_wpad=None,
    scale=1.0,
    fontsize=7,
):
    """Plot Pareto fronts per dataset in a FacetGrid, with optional labels, highlights and first-principles reference points."""

    name_remap = name_remap or {}
    highlight_algorithms = highlight_algorithms or []

    levels_to_show = np.minimum(levels_to_show, len(data['pareto_rank'].unique()))

    cmap = sns.color_palette("viridis", n_colors=levels_to_show)
    g = sns.FacetGrid(
        data,
        col=facet_column,
        height=height,
        aspect=aspect,
        col_wrap=col_wrap,
        margin_titles=True,
        sharex=False,
        sharey=False,
    )

    pareto_optimal_solutions = []
    def custom_plot(data, **kwargs):
        ax = sns.scatterplot(
            data=data[data['pareto_rank'] < levels_to_show],
            x=xcol,
            y=ycol,
            hue='pareto_rank',
            s=120 * scale**2,
            palette=cmap,
            legend=False,
            ax=kwargs.get('ax'),
        )

        styles = ['-', '-.', '--'] + [':']*np.maximum(0, levels_to_show-3)
        for level in range(levels_to_show):
            pf_points = data[data['pareto_rank'] == level].sort_values(by=[xcol, ycol])
            if len(pf_points):
                ax.plot(pf_points[xcol], pf_points[ycol], styles[level % len(styles)] + 'k',
                        alpha=0.6, lw=2 * scale, zorder=1)

        # bootstrap CIs are always computed so the autoscale is the same with or without plot_error
        for alg, sub in data.groupby('algorithm'):
            subset = df_sum[
                (df_sum['algorithm'] == alg) &
                (df_sum[facet_column] == sub[facet_column].iloc[0])
            ]
            if len(subset) <= 1:
                continue

            if sub['pareto_rank'].iloc[0] >= levels_to_show:
                continue

            _, sdx, ciux, cilx = bootstrap(subset[xcol].values, fn=estimator, n=1000)
            _, sdy, ciuy, cily = bootstrap(subset[ycol].values, fn=estimator, n=1000)

            color = cmap[int(sub['pareto_rank'].iloc[0])]
            xval, yval = sub[xcol].iloc[0], sub[ycol].iloc[0]

            alpha = 0.5 if plot_error else 0.0
            ax.plot([cilx, ciux], [yval, yval], color=color, alpha=alpha, lw=2 * scale, zorder=-10)
            ax.plot([xval, xval], [cily, ciuy], color=color, alpha=alpha, lw=2 * scale, zorder=-10)

        ds_name = data[facet_column].values[0]
        if 'fs_data' in globals() and ds_name in fs_data:
            truth = fs_data[ds_name]
            if xcol in truth and ycol in truth:
                ax.scatter(truth[xcol], truth[ycol],
                           marker='*',
                           color='black',
                           s=180 * scale**2,
                           linewidth=1.5 * scale,
                           alpha=0.9,
                           zorder=5)
                if 'latex_eq' in truth:
                    ax.text(
                        0.05, 0.9, truth['latex_eq'],
                        color="k", fontsize=fontsize,
                        horizontalalignment="left", verticalalignment="top",
                        transform=ax.transAxes,
                        bbox=dict(facecolor='w', edgecolor='b', boxstyle='round', alpha=0.5)
                    )

        ta_x, ta_y, ta_text, ta_algo = [], [], [], []
        for idx, row in data.iterrows():
            if row['pareto_rank'] < levels_to_show:
                ta_x.append(row[xcol])
                ta_y.append(row[ycol])
                ta_algo.append(row['algorithm'])
                ta_text.append(name_remap.get(row['algorithm'], row['algorithm']))

        for highlight_algorithm in highlight_algorithms:
            highlight_rows = data[data['algorithm'] == highlight_algorithm]
            if len(highlight_rows) > 0:
                row = highlight_rows.iloc[0]
                in_front = row['pareto_rank'] < levels_to_show
                marker_color = 'tab:green' if in_front else 'tab:red'
                ax.scatter(row[xcol], row[ycol],
                           marker='D', color=marker_color, s=60 * scale**2,
                           edgecolors='k', linewidths=0.5 * scale, zorder=7)
                if not in_front:
                    ta_x.append(row[xcol])
                    ta_y.append(row[ycol])
                    ta_algo.append(highlight_algorithm)
                    ta_text.append(name_remap.get(highlight_algorithm, highlight_algorithm))

        if 'fs_data' in globals() and ds_name in fs_data and len(ta_x) > 0:
            t_x = fs_data[ds_name][xcol]
            t_y = fs_data[ds_name][ycol]

            x_min, x_max = min(ta_x), max(ta_x)
            y_min, y_max = min(ta_y), max(ta_y)
            if x_max == x_min:
                norm_x = np.zeros(len(ta_x))
                norm_tx = 0.0
            else:
                norm_x = [(xi - x_min) / (x_max - x_min) for xi in ta_x]
                norm_tx = (t_x - x_min) / (x_max - x_min)
            if y_max == y_min:
                norm_y = np.zeros(len(ta_y))
                norm_ty = 0.0
            else:
                norm_y = [(yi - y_min) / (y_max - y_min) for yi in ta_y]
                norm_ty = (t_y - y_min) / (y_max - y_min)

            distances = np.sqrt((np.array(norm_x) - norm_tx) ** 2 + (np.array(norm_y) - norm_ty) ** 2)
            closest_idx = int(np.argmin(distances))
            if ds_name == 'first_principles_bode' and len(distances) > 1:
                closest_idx = int(np.argsort(distances)[1])

            chosen_alg = ta_algo[closest_idx]
            try:
                symbolic_model = str(df_sum[(df_sum['algorithm'] == chosen_alg) & (df_sum[facet_column] == ds_name)]['symbolic_model'].values[-1])
            except Exception:
                symbolic_model = ''

            pareto_optimal_solutions.append({
                'algorithm': chosen_alg,
                'dataset': ds_name,
                xcol: ta_x[closest_idx],
                ycol: ta_y[closest_idx],
                'symbolic_model': symbolic_model
            })

            ta_text[closest_idx] = ta_text[closest_idx] + '*'

        if 'ta' in globals():
            try:
                ta.allocate(
                    ax, ta_x, ta_y, ta_text,
                    min_distance=0.04 if ds_name != 'first_principles_supernovae_zr' else 0.08,
                    avoid_label_lines_overlap=True,
                    avoid_crossing_label_lines=True,
                    x_scatter=np.array(ta_x) + ( [fs_data[ds_name][xcol]] if ('fs_data' in globals() and ds_name in fs_data) else [] ),
                    y_scatter=np.array(ta_y) + ( [fs_data[ds_name][ycol]] if ('fs_data' in globals() and ds_name in fs_data) else []),
                    textsize=fontsize, rotation=0,
                    fontweight=['bold' if 'RASR' in t else 'normal' for t in ta_text],
                    **{"bbox": dict(facecolor='w', edgecolor='b', boxstyle='round', alpha=0.5)}
                )
            except Exception:
                for xx, yy, tt in zip(ta_x, ta_y, ta_text):
                    ax.text(xx, yy, tt, fontsize=fontsize, rotation=-20, ha='center', va='bottom',
                            fontweight='bold' if 'RASR' in tt else 'normal')
        else:
            for xx, yy, tt in zip(ta_x, ta_y, ta_text):
                ax.text(xx, yy, tt, fontsize=fontsize, rotation=-20, ha='center', va='bottom',
                        fontweight='bold' if 'RASR' in tt else 'normal')

        ax.set_ylim(0, None)
        ax.grid(ls=":")
        return ax

    g.map_dataframe(custom_plot, col=facet_column)

    auto_ylabel = ycol.replace('_', ' ').title()
    auto_xlabel = xcol.replace('_', ' ').replace('zero ', '').title()
    for i, (title, ax) in enumerate(g.axes_dict.items()):
        ax.set_title(title.replace('first_principles_', '').replace('_', ' ').title())
        if i % col_wrap == 0:
            ax.set_ylabel(ylabel if ylabel is not None else auto_ylabel)
        else:
            ax.set_ylabel('')
        if i >= len(g.axes_dict) - col_wrap:
            ax.set_xlabel(xlabel if xlabel is not None else auto_xlabel)
        else:
            ax.set_xlabel('')

    plt.tight_layout()
    save(name=f"pareto_facets_{xcol}_{ycol}", pad=save_pad, w_pad=save_wpad)
    return g, pareto_optimal_solutions


df_results_plot = df_results.copy()
df_results_plot = df_results_plot[
    ~df_results_plot['algorithm'].isin(['Genetic Engine rs', 'Genetic Engine hc', 'Genetic Engine 1p1'])
]
df_results_plot = df_results_plot[
    ~df_results_plot['dataset'].isin(['first_principles_supernovae_zg'])
]
df_results_plot = df_results_plot[df_results_plot['model_size'] > 1]

best_idx = df_results_plot.groupby(['algorithm', 'dataset'])['r2_zero_test'].transform(max) == df_results_plot['r2_zero_test']
df_results_plot = df_results_plot[best_idx]

pareto_data = prepare_pareto_data(
    df_results_plot,
    estimator=np.median,
    xcol='r2_zero_test',
    ycol='model_size'
)

g, pareto_optimal_solutions = plot_pareto_facets(
    df_sum=df_results_plot,
    data=pareto_data,
    estimator=np.median,
    plot_error=True,
    levels_to_show=1,
    xcol='r2_zero_test',
    ycol='model_size',
    name_remap=algorithm_name_remap,
    highlight_algorithms=highlight_algorithms,
)

In [ ]:

df_results_median = df_results.copy()
df_results_median = df_results_median[
    ~df_results_median['algorithm'].isin(['Genetic Engine rs', 'Genetic Engine hc', 'Genetic Engine 1p1'])
]
df_results_median = df_results_median[
    ~df_results_median['dataset'].isin(['first_principles_supernovae_zg'])
]
df_results_median = df_results_median[df_results_median['model_size'] > 1]

pareto_data_median = prepare_pareto_data(
    df_results_median,
    estimator=np.median,
    xcol='r2_zero_test',
    ycol='model_size'
)

# 4 rows x 3 columns of square facets at 6pt, spanning the 5.5in text width
_rc = {
    'font.size': 6, 'axes.titlesize': 6, 'axes.labelsize': 6,
    'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'mathtext.fontset': 'stix',
}
with plt.rc_context(_rc):
    g_median, pareto_optimal_solutions_median = plot_pareto_facets(
        df_sum=df_results_median,
        data=pareto_data_median,
        estimator=np.median,
        plot_error=False,
        levels_to_show=1,
        xcol='r2_zero_test',
        ycol='model_size',
        name_remap=algorithm_name_remap,
        highlight_algorithms=highlight_algorithms,
        col_wrap=3,
        xlabel='Median Test $R^2$',
        ylabel='Median Expression Size',
        height=1.83,
        aspect=1,
        scale=0.5,
        fontsize=6,
    )


In [ ]:

compare_dataset = 'first_principles_supernovae_zr'
compare_algorithm = 'Operon'
n_seeds = 5
sample_random_state = 0

for alg in list(highlight_algorithms) + [compare_algorithm]:
    label = algorithm_name_remap.get(alg, alg)
    subset = df_results[
        (df_results['algorithm'] == alg) &
        (df_results['dataset'] == compare_dataset)
    ]
    print(f"=== {label} ({len(subset)} seeds available) ===")
    sample = subset.sample(n=min(n_seeds, len(subset)), random_state=sample_random_state)
    for _, row in sample.sort_values('random_state').iterrows():
        print(f"  seed {row['random_state']}: r2_zero_test={row['r2_zero_test']:.4f}, model_size={row['model_size']}")
        print(f"    {row['symbolic_model']}")
    print()


In [ ]:
pareto_data.to_csv('../figs/first-principles-tuning/pareto_data.csv', index=False)
display(pareto_data)

pareto_optimal_solutions = pd.DataFrame().from_records(pareto_optimal_solutions)
pareto_optimal_solutions.to_csv('../figs/first-principles-tuning/pareto_optimal_solutions.csv', index=False)
display(pareto_optimal_solutions)

In [ ]:
pareto_optimal_solutions_df = pd.DataFrame(pareto_optimal_solutions)
with pd.option_context("max_colwidth", 1000):
    print(pareto_optimal_solutions_df.replace({'first_principles_':''}, regex=True).sort_values('dataset').round(2).style.to_latex())